### 토큰 속도 측정
- 파일 : nA2_token_latency.ipynb
- 역할 : 소형 언어 모델을 실행해 TTFT, TPOT, tokens/sec를 직접 측정합니다.
- 기능 :
    1. 모델과 토크나이저 준비  
    2. TTFT 측정  
    3. TPOT와 tokens/sec 계산  
    4. 입력 길이에 따른 TTFT와 TPOT  
    5. 배치 처리의 효과  


# 2장 토큰 처리량과 지연: 소형 언어 모델의 토큰 속도 측정

부록 원고의 코드와 동일합니다. 핵심 수식은 코드의 주석에 함께 적어 두었습니다.

>> [코드 A2-1] 모델과 토크나이저 준비 (nA2_token_latency.ipynb, STEP 1)
<hr>

In [ ]:
#- 주요 기능 : 토큰 지연시간 측정을 위해 언어모델과 토크나이저를 로드하고 추론 환경 준비

#- 패키지 설치
!pip install -q transformers accelerate

#- 모듈 로딩
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#- 모델·토크나이저 로딩 (FP32)
MODEL_ID = "Qwen/Qwen2.5-0.5B-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float32)
model.eval()
print("파라미터 수:", sum(p.numel() for p in model.parameters()))

# 확인 포인트 --------------------------------------------------------------------------
# - 파라미터 수 체크

>> [코드 A2-2] TTFT 측정 (STEP 2)
<hr>

In [ ]:
#- 주요 기능 : 프롬프트 입력 후 첫 토큰이 생성될 때까지의 시간인 TTFT 측정

#- 짧은 프롬프트를 채팅 형식으로 인코딩
prompt = "온디바이스 AI의 장점을 세 문장으로 설명해 주세요."
msgs = [{"role": "user", "content": prompt}]

#- 채팅 형식 + 텐서로 인코딩
enc = tok.apply_chat_template(msgs, add_generation_prompt=True,
                              return_dict=True, return_tensors="pt")
print("입력 토큰 수:", enc["input_ids"].shape[1])

#- 첫 토큰까지의 시간(TTFT) 측정 (빈칸 채우기)
t0 = time.perf_counter()
with torch.no_grad():
    _ = model.generate(**enc, max_new_tokens=____, do_sample=False)
ttft = time.perf_counter() - t0
print(f"TTFT: {ttft:.3f}초")

# 확인 포인트 --------------------------------------------------------------------------
# - 같은 셀 2회 실행 시 두 번째 속도 향상 체크 (준비 비용 제외)

>> [코드 A2-3] TPOT와 tokens/sec 계산 (STEP 3)
<hr>

In [ ]:
#- 최대 64토큰 생성의 총 시간 측정 (교육용 근사: 별도 실행으로 TTFT·TPOT를 근사)
n_new = 64
t0 = time.perf_counter()
with torch.no_grad():
    out = model.generate(**enc, max_new_tokens=n_new, do_sample=False)
total = time.perf_counter() - t0

#- TPOT와 tokens/sec 계산
# 총 시간 ≈ TTFT + (n_new − 1) × TPOT  →  TPOT = (총 시간 − TTFT) / (n_new − 1)
tpot = (total - ttft) / (n_new - 1)
decode_tps = 1 / tpot
print(f"총 {total:.2f}초, TPOT {tpot*1000:.0f}ms, Decode {decode_tps:.1f} tokens/sec")

# 확인 포인트 ------------------------------------------------------------------------
# - 총 시간 > TTFT 체크
# - decode_tps 값 기록 (프로세서별 토큰 효율에서 CPU 공개 실측과 비교)

>> [코드 A2-4] 입력 길이에 따른 TTFT와 TPOT (STEP 4)
<hr>

In [ ]:
#- 입력을 수십 배 길게 만든 프롬프트
long_prompt = "다음 문장을 참고하세요. " + "온디바이스 AI는 기기 안에서 추론합니다. " * 40 \
              + "이제 온디바이스 AI의 장점을 세 문장으로 설명해 주세요."
msgs_l = [{"role": "user", "content": long_prompt}]
enc_l = tok.apply_chat_template(msgs_l, add_generation_prompt=True,
                                return_dict=True, return_tensors="pt")
print("입력 토큰 수:", enc_l["input_ids"].shape[1])

#- 긴 입력의 TTFT와 TPOT를 같은 방식으로 측정
t0 = time.perf_counter()
with torch.no_grad():
    _ = model.generate(**enc_l, max_new_tokens=1, do_sample=False)
ttft_l = time.perf_counter() - t0

t0 = time.perf_counter()
with torch.no_grad():
    # 64토큰 생성 총 시간
    _ = model.generate(**enc_l, max_new_tokens=n_new, do_sample=False)
total_l = time.perf_counter() - t0
tpot_l = (total_l - ttft_l) / (n_new - 1)
print(f"짧은 입력: TTFT {ttft:.3f}초, TPOT {tpot*1000:.0f}ms")
print(f"긴 입력:  TTFT {ttft_l:.3f}초, TPOT {tpot_l*1000:.0f}ms")

# 확인 포인트 -------------------------------------------------------------------------
# - 입력 길이 배수와 TTFT 배수 비교
# - TPOT 증가 폭 확인 (KV 캐시 읽기의 영향)

>> [코드 A2-5] 배치 처리의 효과 (STEP 5)
<hr>

In [ ]:
#- 요청 4개를 배치 하나로
prompts = ["NPU란 무엇인가요?", "양자화의 장점은?",
           "엣지 컴퓨팅의 예를 들어 주세요.", "KV 캐시는 왜 필요한가요?"]

# 요청마다 채팅 형식 문자열로
chats = [tok.apply_chat_template([{"role": "user", "content": p}],
         add_generation_prompt=True, tokenize=False) for p in prompts]

# 길이가 다른 입력을 왼쪽 패딩으로 정렬
tok.padding_side = "left"
enc_b = tok(chats, return_tensors="pt", padding=True)

#- 배치 생성의 총 시간과 장비 전체 처리량
t0 = time.perf_counter()
with torch.no_grad():
    _ = model.generate(**enc_b, max_new_tokens=32, do_sample=False)
total_b = time.perf_counter() - t0

# 4요청 × 32토큰 / 총 시간
batch_tps = 4 * 32 / total_b
print(f"배치 4: 총 {total_b:.2f}초, 장비 전체 {batch_tps:.1f} tokens/sec")

# 확인 포인트 -------------------------------------------------------------------------
# - 장비 전체 tokens/sec > STEP 3 decode_tps 체크